# Stockformer S&P 500 — Training on Colab

This notebook runs the full pipeline: data generation → training → inference → evaluation.

**Requirements:** T4 GPU runtime (Runtime → Change runtime type → T4 GPU)

In [1]:
# 1. Clone repo and install dependencies
!git clone https://github.com/GonAlonsoLid/tfm-stockformer.git
%cd tfm-stockformer
!git checkout develop

Cloning into 'tfm-stockformer'...
remote: Enumerating objects: 1112, done.
remote: Counting objects: 100% (1112/1112), done.
remote: Compressing objects: 100% (509/509), done.
remote: Total 1112 (delta 646), reused 1034 (delta 575), pack-reused 0 (from 0)
Receiving objects: 100% (1112/1112), 4.91 MiB | 11.58 MiB/s, done.
Resolving deltas: 100% (646/646), done.
/content/tfm-stockformer
Branch 'develop' set up to track remote branch 'develop' from 'origin'.
Switched to a new branch 'develop'


In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt
!pip install -q lightgbm>=4.0
# GraphEmbedding for Struc2Vec (step 4 of pipeline)
!pip install -q git+https://github.com/shenweichen/GraphEmbedding.git --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 5.5 MB/s eta 0:00:00


In [ ]:
# 3. Run data pipeline (download OHLCV, build labels, graph, Alpha360)
# Takes ~10-15 min on first run; skips completed steps on re-run
!python scripts/build_pipeline.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# 4. Build extended features
!python scripts/build_alpha158.py --data_dir data/Stock_SP500_2018-01-01_2026-03-16
!python scripts/build_macro_features.py --data_dir data/Stock_SP500_2018-01-01_2026-03-16

In [ ]:
# 5. Audit data integrity
!python scripts/audit_alignment.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# 6. LightGBM baseline (establishes IC ceiling with current features)
!python scripts/run_lightgbm_baseline.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# 7. Train Stockformer (ranking loss + IC tracking + early stopping)
# Monitor with TensorBoard: look for Val/IC_Spearman
%load_ext tensorboard
%tensorboard --logdir runs/

In [ ]:
# Full model (layers=2, dims=128)
!python MultiTask_Stockformer_train.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# Reduced model for A/B test (layers=1, dims=64)
!python MultiTask_Stockformer_train.py --config config/Multitask_Stock_SP500_reduced.conf

In [ ]:
# 8. Inference on test set
!python scripts/run_inference.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# 9. Compute IC
!python scripts/compute_ic.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# 10. Backtest — all three strategies
OUTPUT_DIR = "output/Multitask_output_SP500_2018-01-01_2026-03-16"

!python scripts/run_backtest.py --output_dir {OUTPUT_DIR} \
    --config config/Multitask_Stock_SP500.conf --strategy longonly

!python scripts/run_backtest.py --output_dir {OUTPUT_DIR} \
    --config config/Multitask_Stock_SP500.conf --strategy longshort

!python scripts/run_backtest.py --output_dir {OUTPUT_DIR} \
    --config config/Multitask_Stock_SP500.conf --strategy longshort_hedged

In [ ]:
# 11. Walk-forward evaluation (robust IC across rolling windows)
!python scripts/run_walkforward.py --config config/Multitask_Stock_SP500.conf

In [ ]:
# 12. Display results
import pandas as pd
from IPython.display import display

print("=" * 60)
print("LightGBM Baseline")
print("=" * 60)
display(pd.read_csv("output/lightgbm_baseline/evaluation_summary.csv"))

print("\n" + "=" * 60)
print("Backtest Summary")
print("=" * 60)
display(pd.read_csv(f"{OUTPUT_DIR}/backtest_summary.csv"))

print("\n" + "=" * 60)
print("Walk-Forward Summary")
print("=" * 60)
try:
    display(pd.read_csv("output/walkforward/walkforward_summary.csv"))
except FileNotFoundError:
    print("Not yet available")

In [ ]:
# 13. Show equity curve
from IPython.display import Image
Image(f"{OUTPUT_DIR}/equity_curve.png")